# Week 18: Autonomous AI Agents & Multi-Agent Systems
## Notebook 09: Reflexion: Verbal Reinforcement Learning & Self-Reflection

### 1. Conceptual Intuition & Theoretical Background
Traditional Reinforcement Learning (PPO, Q-Learning) optimizes policies by updating model weights via numerical scalar rewards $R \in \mathbb{R}$. However, scalar rewards provide weak, uninformative credit assignment: knowing reward was $-0.5$ does not tell an LLM **why** it failed or **what specific code edit** is needed.

**Reflexion** introduces **Verbal Reinforcement Learning**:
Instead of computing weight gradients, the agent updates its policy by generating natural language **self-reflections** that are stored in an **episodic verbal memory buffer** $\mathcal{M}$:

1. **Actor ($ heta$):** Proposes trajectory $\tau_t = (a_1, o_1, \dots, a_T, o_T)$.
2. **Evaluator:** Evaluates trial outcome against objective test suites (e.g. Unit Test Pass/Fail).
3. **Self-Reflection Generator:** Reflects on trajectory failure:
$$r_t \sim \text{LLM}\left( \text{Trajectory } \tau_t,\ \text{Evaluation Trace},\ \mathcal{M}_{t-1} \right)$$
4. **Episodic Memory Update:**
$$\mathcal{M}_t = \mathcal{M}_{t-1} \cup \{r_t\}$$
On the subsequent trial, verbal memory reflections are prepended to the prompt, steering the agent away from previous failure modes.

---

### 2. Scratch Python Implementation: Reflexion Code Repair Loop

In [ ]:
class ReflexionCodeAgentScratch:
    def __init__(self):
        self.episodic_memory = [] # Buffer of past reflections
        
    def actor_generate_code(self, task_spec, trial):
        """Actor generates code, conditioned on previous self-reflections."""
        print(f"--- Trial {trial} Code Generation ---")
        if not self.episodic_memory:
            # First attempt: common off-by-one bug
            return "def sum_to_n(n):\n    return sum(range(n))"
        else:
            print("  [In-Context Verbal Guidance]")
            for r in self.episodic_memory:
                print(f"  Reflection: '{r}'")
            # Corrected attempt
            return "def sum_to_n(n):\n    return sum(range(1, n + 1))"
            
    def evaluator_unit_test(self, code_str):
        """Evaluator runs assertions against generated code."""
        local_scope = {}
        try:
            exec(code_str, {}, local_scope)
            fn = local_scope['sum_to_n']
            # Test: sum_to_n(5) = 1 + 2 + 3 + 4 + 5 = 15
            res = fn(5)
            if res == 15:
                return True, "All unit tests passed!"
            else:
                return False, f"AssertionError: Expected 15, but got {res}."
        except Exception as e:
            return False, f"Runtime error: {str(e)}"
            
    def reflection_generator(self, failed_code, error_trace):
        """Generates semantic verbal critique explaining failure."""
        reflection = (
            f"Failed code used 'range(n)' which excludes n and includes 0. "
            f"Fix: Use 'range(1, n + 1)' to include the boundary value."
        )
        self.episodic_memory.append(reflection)
        return reflection

agent = ReflexionCodeAgentScratch()
task = "Write a function sum_to_n(n) returning the sum of integers from 1 to n."

max_trials = 3
for trial in range(1, max_trials + 1):
    candidate_code = agent.actor_generate_code(task, trial)
    passed, feedback = agent.evaluator_unit_test(candidate_code)
    
    if passed:
        print(f"\n✅ SUCCESS at Trial {trial}!")
        print(f"Verified Solution:\n{candidate_code}")
        break
    else:
        print(f"❌ FAILED at Trial {trial}. Error: {feedback}")
        refl = agent.reflection_generator(candidate_code, feedback)
        print(f"Recorded Verbal Reflection: '{refl}'\n")

### 3. Step-by-Step Code Walkthrough
1. **Verbal Feedback Loop**: Evaluator unit test failure triggers automated semantic self-reflection generation.
2. **Episodic Context Injection**: Past failure analyses are injected into the agent's context window.
3. **Weightless Adaptation**: Resolves errors without fine-tuning model parameters.

---

### 4. Industry Applications & Trade-offs
- **Autonomous Software Engineering (SWE-bench):** Self-healing code generation bots diagnosing pytest failures.
- **Trade-off:** Verbal memory buffers accumulate tokens, increasing prompt costs over long debugging loops.

---

### 5. Top Interview Questions & Answers
**Q1: How does Verbal Reinforcement Learning differ from traditional Policy Gradient RL?**
> **Answer:** Policy gradient algorithms update neural network weights via scalar rewards ($R$). Verbal RL keeps weights frozen and updates an episodic text memory buffer with linguistic critiques, guiding subsequent attempts through in-context learning.

**Q2: What prevents Reflexion agents from repeating the same mistake across trials?**
> **Answer:** Explicit verbal memory strings injected into the prompt describe the root cause of past failures and recommend corrective strategies, acting as negative constraint guidance.